# jlenskit quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamC1249/j-lens/blob/main/examples/quickstart.ipynb)

Fit a **Jacobian lens** on GPT-2, read out the tokens each hidden layer is *disposed to say*, compare it to the classic **logit lens**, and render an interactive viewer — in a few minutes on CPU.

> The J-lens comes from Anthropic's [*Verbalizable Representations Form a Global Workspace in Language Models*](https://transformer-circuits.pub/2026/workspace/index.html). It's a principled refinement of the logit lens: instead of assuming a hidden state already lives in the final-layer basis, it *transports* the state there via the per-layer Jacobian `J_ℓ`, then decodes with the model's own unembedding.

In [ ]:
# On Colab, install from GitHub. Locally, `pip install -e .` from a clone instead.
%pip install -q git+https://github.com/SamC1249/j-lens.git

## 1. Load any HuggingFace decoder

`jlenskit.load` autodetects the architecture and wraps it in an `Adapter` — the single interface the lens math talks to. GPT-2 (124M) runs comfortably on CPU.

In [ ]:
import jlenskit
from jlenskit.core import JacobianLens
from jlenskit.data import load_corpus

adapter = jlenskit.load("gpt2")
print(f"{adapter.n_layers} layers, d_model={adapter.d_model}, vocab={adapter.vocab_size}")

## 2. Fit the lens

Fitting estimates the corpus-averaged Jacobian `J_ℓ` for every layer. Thanks to autoregressive causality this costs only ~`d_model` backward passes over the corpus — independent of sequence length.

In [ ]:
batches = load_corpus(
    {"source": "builtin", "seq_len": 64, "n_sequences": 32}, adapter.tokenizer
)
lens = JacobianLens.fit(adapter, batches, chunk_size=64, seed=0, progress=True)
print(f"fitted layers: {lens.layers}")

## 3. Apply it — watch the answer crystallize across layers

Later layers should converge on the model's actual prediction; earlier layers reveal the "silent" tokens forming underneath.

In [ ]:
prompt = "The Eiffel Tower is located in the city of"
res = lens.apply(adapter, prompt, positions=[-1], top_k=5)
for dl in res.decoded[0]:
    print(f"layer {dl.layer:2d}: {[t.strip() for t in dl.tokens]}")

## 4. Compare against the logit lens

Same activation, same unembedding — the only difference is whether we transport through `J_ℓ` first. The J-lens should surface coherent tokens in *earlier* layers than the logit lens.

In [ ]:
from jlenskit import logit_lens

base = logit_lens(adapter, prompt, positions=[-1], top_k=5)
print(f"{'layer':>5} | {'logit lens':<28} | J-lens")
print("-" * 70)
for jl, bl in zip(res.decoded[0], base.decoded[0]):
    j = ", ".join(t.strip() for t in jl.tokens[:3])
    b = ", ".join(t.strip() for t in bl.tokens[:3])
    print(f"{jl.layer:>5} | {b:<28} | {j}")

## 5. Visualize

`viz.render` writes a self-contained, offline HTML file (all CSS inlined). Here we render it inline in the notebook.

In [ ]:
from IPython.display import HTML, display

from jlenskit import viz

multi = lens.apply(adapter, prompt, positions=list(range(-4, 0)), top_k=5)
path = viz.render(multi, "quickstart_viz.html", kind="grid")
display(HTML(path.read_text()))

## Where to go next

- **Metrics** — `jlenskit.metrics.evaluate` scores top-k accuracy / kurtosis / autocorrelation per layer to locate the "workspace" layers.
- **Interventions** — `jlenskit.jspace` decomposes an activation into lens vectors and does causal `inject` / `swap` experiments.
- **CLI** — everything here is also a one-line reproducible command: `jlenskit fit|apply|eval|viz|intervene CONFIG.yaml`.
- **Bigger models** — swap `"gpt2"` for `"Qwen/Qwen2-0.5B"`, `"meta-llama/Llama-3.2-1B"`, etc. Use a GPU for anything past ~0.5B.